# Bloco 2: Baselines clássicos
**Antes de usar um LLM, quanto se acerta sem ele?**

1. **Dicionário**: regras explícitas (regex). Barato, transparente, auditável.
2. **TF-IDF + regressão logística multinomial** com penalização, avaliada por validação cruzada na amostra-ouro (n = 120).

Mensagem: um LLM só "vale a pena" se superar esses baselines de forma consistente, e isso é uma afirmação estatística, não uma impressão.

In [1]:
# === Configuração: funciona localmente e no Google Colab ===
import os, sys, subprocess
from pathlib import Path

# PROVISÓRIO: endereço do repositório público do minicurso (ajustar quando publicado)
REPO_URL = "https://github.com/jodavid/minicurso-ia-medicao"

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    if not Path("minicurso-ia-medicao").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    RAIZ = Path("minicurso-ia-medicao").resolve()
else:
    # localmente: a raiz é a pasta do minicurso (este notebook fica em python/)
    RAIZ = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "python"))
SAIDA = RAIZ / "resultados" / "python"          # o R grava em resultados/; o Python, aqui
(SAIDA / "figuras").mkdir(parents=True, exist_ok=True)
(SAIDA / "tabelas").mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: ~/minicurso_IA_medicao_publico


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from medicao import (CATEGORIAS, STOPWORDS_PT, carrega_corpus, carrega_ouro, classifica_dicionario,
                     eleitoral_dicionario, resumo_metricas)

corpus = carrega_corpus(RAIZ)
ouro = carrega_ouro(RAIZ).merge(corpus[["doc_id", "ementa"]], on="doc_id")
ouro["categoria"].value_counts()

AVISO: a amostra-ouro ainda NÃO foi revisada por humanos (coluna `revisado`). Os resultados são ilustrativos.


categoria
REGULAR     69
PUNIR       22
PROMOVER    16
OUTRO       13
Name: count, dtype: int64

## 1. Dicionário

**Atenção (ponto para discussão):** as regras foram escritas depois de **ler** as ementas do corpus. O dicionário "viu a prova", e seu desempenho aqui é otimista. Num estudo real, as regras são definidas antes ou validadas em dados novos (ex.: proposições apresentadas depois da data de coleta).

In [3]:
ouro["pred_dic"] = ouro["ementa"].map(classifica_dicionario)
print(resumo_metricas(ouro["categoria"], ouro["pred_dic"]))
pd.crosstab(ouro["categoria"], ouro["pred_dic"], rownames=["humano"], colnames=["dicionário"])

{'n': 120, 'falhas_formato': 0.0, 'acuracia': 0.8083333333333333, 'f1_macro': 0.7815508135275577, 'kappa': 0.698887191795767}


dicionário,OUTRO,PROMOVER,PUNIR,REGULAR
humano,,,,
OUTRO,11,1,0,1
PROMOVER,0,11,0,5
PUNIR,2,0,20,0
REGULAR,2,11,1,55


## 2. TF-IDF + regressão logística (validação cruzada 10-fold)

As previsões são *out-of-fold*: cada ementa é classificada por um modelo que **não** a viu no treino. O parâmetro de regularização é escolhido por uma validação cruzada interna (5-fold).

In [4]:
modelo = make_pipeline(
    TfidfVectorizer(lowercase=True, stop_words=STOPWORDS_PT, min_df=2,
                    token_pattern=r"(?u)\b[^\W\d_]{2,}\b"),
    LogisticRegressionCV(Cs=10, cv=5, max_iter=5000, scoring="accuracy"),
)
dobras = StratifiedKFold(n_splits=10, shuffle=True, random_state=2026)
ouro["pred_ml"] = cross_val_predict(modelo, ouro["ementa"], ouro["categoria"], cv=dobras)

maj = pd.Series(["REGULAR"] * len(ouro))
baselines = pd.DataFrame([
    {"metodo": "Dicionário (regex)", **resumo_metricas(ouro["categoria"], ouro["pred_dic"])},
    {"metodo": "TF-IDF + logística (CV 10)", **resumo_metricas(ouro["categoria"], ouro["pred_ml"])},
    {"metodo": "Classe majoritária (REGULAR)", **resumo_metricas(ouro["categoria"], maj)},
])
baselines.round(3)

,metodo,n,falhas_formato,acuracia,f1_macro,kappa
0,Dicionário (regex),120,0.0,0.808,0.782,0.699
1,TF-IDF + logística (CV 10),120,0.0,0.725,0.565,0.508
2,Classe majoritária (REGULAR),120,0.0,0.575,0.183,0.000


In [5]:
baselines.to_csv(SAIDA / "tabelas/baselines.csv", index=False)
ouro[["doc_id", "categoria", "pred_dic", "pred_ml"]].to_csv(SAIDA / "tabelas/predicoes_baselines.csv", index=False)

# Dicionário aplicado a TODO o corpus (usado como "instrumento barato" no Bloco 5)
pd.DataFrame({"doc_id": corpus["doc_id"],
              "categoria_dic": corpus["ementa"].map(classifica_dicionario),
              "eleitoral_dic": corpus["ementa"].map(eleitoral_dicionario)}
             ).to_csv(SAIDA / "tabelas/dicionario_corpus.csv", index=False)

### Para discutir
* Por que o modelo supervisionado, com 120 exemplos, fica **abaixo** do dicionário?
* Quantos exemplos rotulados seriam necessários para ele superar as regras? Quanto custa produzi-los?
* Que erros do dicionário são **sistemáticos** (sempre na mesma direção)?